In [0]:
from scr.endpoints import list_keys_all, list_keys_changes, items_many, next_offset
from scr.connection import APIClient
import requests
import time
import json

BASE = "https://productapi-synkka.gs1.fi"
EMAIL = dbutils.secrets.get("gs1-kv", "email")
PASSWORD = dbutils.secrets.get("gs1-kv", "password")
GLN = dbutils.secrets.get("gs1-kv", "gln")

KEY_BATCH_SIZE = 90000
SINCE_ISO = "2025-09-10T07:03:26.1655687Z"

client = APIClient(BASE, EMAIL, PASSWORD, GLN)
client.authenticate(verbose=True)   

# Testaa /All
resp_all = list_keys_all(client, batch_size=KEY_BATCH_SIZE)
all_items = resp_all.get("Items") or []
print("ALL count:", len(all_items))
if all_items:
    print("ALL first item:\n", json.dumps(all_items[0], indent=2, ensure_ascii=False))

# Testaa /Changes vain 'Since':lla
resp_chg = list_keys_changes(client, batch_size=KEY_BATCH_SIZE, since=SINCE_ISO)
chg_items = resp_chg.get("Items") or []
print("\nCHANGES count:", len(chg_items))
if chg_items:
    print("CHANGES first item:\n", json.dumps(chg_items[0], indent=2, ensure_ascii=False))

time.sleep(1.2)   

# Testaa /Many muutamalle Id:lle (ensin changesistä, muuten all:sta)
sample_ids = [it.get("Id") for it in (chg_items or all_items) if it.get("Id")][:5]
if sample_ids:
    resp_many = items_many(client, sample_ids)  # oletuspolku: /SdpCatalogueItem/Many
    prod_items = resp_many.get("Items") or []
    print("\nMANY items:", len(prod_items))
    if prod_items:
        print("MANY first product:\n", json.dumps(prod_items[0], indent=2, ensure_ascii=False))
else:
    print("\nMANY: ei Id-näytteitä")

